# Import libraries

In [125]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score


# Import stport history data and feature selection

In [126]:
#read data 
df= pd.read_csv('combined_matches.csv')
df.drop(columns=['Date','HomeGoals','AwayGoals'], inplace=True)
df.head()

,League,HomeTeam,AwayTeam,Result
0,Serie A,Atalanta,Cagliari,H
1,Serie A,Genoa,Roma,H
2,Serie A,Inter,Reggiana,H
3,Serie A,Juventus,Cremonese,H
4,Serie A,Lazio,Foggia,D


In [127]:
#convert columns to lower
df[['League','HomeTeam','AwayTeam', 'Result']]=df[['League','HomeTeam','AwayTeam', 'Result']].applymap(str.lower)
df.head()

C:\Users\Ricmwas\AppData\Local\Temp\ipykernel_17864\33554188.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[['League','HomeTeam','AwayTeam', 'Result']]=df[['League','HomeTeam','AwayTeam', 'Result']].applymap(str.lower)


,League,HomeTeam,AwayTeam,Result
0,serie a,atalanta,cagliari,h
1,serie a,genoa,roma,h
2,serie a,inter,reggiana,h
3,serie a,juventus,cremonese,h
4,serie a,lazio,foggia,d


In [128]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216883 entries, 0 to 216882
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   League    216883 non-null  object
 1   HomeTeam  216883 non-null  object
 2   AwayTeam  216883 non-null  object
 3   Result    216883 non-null  object
dtypes: object(4)
memory usage: 6.6+ MB


# Match the team with sportpesa teams 
Use process.extractOne from thefuzz to match each team in the table above with the closest name in your dataset.
Set a similarity threshold to avoid incorrect matche

In [129]:
from thefuzz import process
# Create the DataFrame (replace this with your actual DataFrame)
df_s = pd.DataFrame({
    
    "Mapped League": [
    "serie a", "la liga 2", "bundesliga 2", "serie a", "la liga 2",
    "english premier league", "serie b", "scottish premier league", "ligue 1", "ligue 1",
    "ligue 1", "serie b", "bundesliga", "english premier league", "la liga",
    "la liga 2", "la liga 2"
],
    "Home Team": [
        "SSC Napoli", "Real Zaragoza", "Karlsruher SC", "AC Monza", "CD Leganes", 
        "Newcastle United", "US Catanzaro", "Aberdeen FC", "AJ Auxerre", "Montpellier HSC",
        "Angers SCO", "SSC Bari", "FC Augsburg", "Manchester United", "RCD Mallorca",
        "Albacete Balompie", "Club Deportivo Eldense"
    ],
    "Away Team": [
        "Inter Milano", "Sporting Gijon", "1. FC Cologne", "Torino FC", "Getafe CF", 
        "Brighton & Hove Albion", "Reggiana 1919", "Dundee United", "Strasbourg Alsace", "Stade Rennes",
        "Toulouse FC", "Sampdoria Genoa", "SC Freiburg", "Fulham FC", "Deportivo Alaves",
        "Cadiz CF", "Levante UD"
    ]
})

df_s.head()

,Mapped League,Home Team,Away Team
0,serie a,SSC Napoli,Inter Milano
1,la liga 2,Real Zaragoza,Sporting Gijon
2,bundesliga 2,Karlsruher SC,1. FC Cologne
3,serie a,AC Monza,Torino FC
4,la liga 2,CD Leganes,Getafe CF


In [130]:

# Create a mapping of table team names to best match in dataset
def map_teams(table_teams, dataset_teams, threshold=80):
    """
    Maps home teams from a given list to the closest match in a dataset using fuzzy matching.

    Parameters:
    table_teams (list): List of team names to map.
    dataset_teams (list or array): List of unique team names from the dataset.
    threshold (int): Minimum match score to accept a match (default is 80).

    Returns:
    dict: A dictionary mapping table team names to their best match in the dataset.
    """
    team_mapping = {}
    for team in table_teams:
        match, score = process.extractOne(team, dataset_teams)
        team_mapping[team] = match if score >= threshold else None
    return team_mapping

team_mapping_home= map_teams(df_s['Home Team'],df['HomeTeam'].unique() )
team_mapping_away= map_teams(df_s['Away Team'],df['AwayTeam'].unique() )
# Apply mapping to the "Home Team" column
df_s["Mapped Home Team"] = df_s["Home Team"].map(team_mapping_home).fillna(df_s["Home Team"])
# Apply mapping to the "Away Team" column (optional)
df_s["Mapped Away Team"] = df_s["Away Team"].map(team_mapping_away).fillna(df_s["Away Team"])
# Display updated DataFrame
df_s=df_s[['Mapped League','Mapped Home Team','Mapped Away Team']]
df_s.columns= df.columns[:3].tolist()
df_s

,League,HomeTeam,AwayTeam
0,serie a,napoli,milan
1,la liga 2,zaragoza,sp gijon
2,bundesliga 2,karlsruhe,fc koln
3,serie a,monza,torino
4,la liga 2,leganes,getafe
5,english premier league,newcastle,brighton
6,serie b,catanzaro,reggiana
7,scottish premier league,aberdeen,dundee united
8,ligue 1,auxerre,strasbourg
9,ligue 1,montpellier,rennes


# Creat the train and test dataset 

In [131]:
# Encode categorical features
label_encoders = {}
for col in ['League', 'HomeTeam', 'AwayTeam']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

#Encode target variable
result_encoder = LabelEncoder()
df['Result'] = result_encoder.fit_transform(df['Result'])

# Split data
X = df.drop(columns=['Result'])
y = df['Result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)



In [132]:
X_train

,League,HomeTeam,AwayTeam
174638,8,528,396
198289,12,90,103
64130,9,301,301
146225,7,140,197
168583,5,503,138
...,...,...,...
119879,0,338,270
103694,19,280,360
131932,2,580,471
146867,3,125,290


# Run the model

In [133]:
# Train XGBoost Model
model = xgb.XGBClassifier(objective='multi:softmax', num_class=3, eval_metric='mlogloss', use_label_encoder=False)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

c:\Users\Ricmwas\Documents\python_projects\Ifiwastodothis\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [16:30:44] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Model Accuracy: 0.47


# Predict the matches

In [134]:


def predict_matches(df, model, label_encoders, result_encoder):
    """
    Predicts match results for an entire DataFrame.

    Parameters:
    df (pd.DataFrame): DataFrame containing columns ['League', 'HomeTeam', 'AwayTeam'].
    model (sklearn model): Trained prediction model.
    label_encoders (dict): Dictionary of trained LabelEncoders for categorical features.
    result_encoder (LabelEncoder): Trained LabelEncoder for result decoding.

    Returns:
    pd.DataFrame: The input DataFrame with an added 'PredictedResult' column.
    """
    df_encoded = df.copy()

    # Encode categorical features
    for col in ['League', 'HomeTeam', 'AwayTeam']:
        if col in label_encoders:
            df_encoded[col] = label_encoders[col].transform(df_encoded[col])

    # Predict match outcomes
    df_encoded['PredictedResult'] = model.predict(df_encoded[['League', 'HomeTeam', 'AwayTeam']])

    # Decode results back to original labels
    df['PredictedResult'] = result_encoder.inverse_transform(df_encoded['PredictedResult'])

    return df

# Example usage
match_df = pd.DataFrame([
    ['serie a', 'juventus', 'roma'],
    ['premier league', 'manchester united', 'chelsea'],
    ['la liga', 'real madrid', 'barcelona']
], columns=['League', 'HomeTeam', 'AwayTeam'])

predicted_matches = predict_matches(df_s, model, label_encoders, result_encoder)
print(predicted_matches)



                     League     HomeTeam       AwayTeam PredictedResult
0                   serie a       napoli          milan               h
1                 la liga 2     zaragoza       sp gijon               h
2              bundesliga 2    karlsruhe        fc koln               h
3                   serie a        monza         torino               h
4                 la liga 2      leganes         getafe               h
5    english premier league    newcastle       brighton               h
6                   serie b    catanzaro       reggiana               h
7   scottish premier league     aberdeen  dundee united               h
8                   ligue 1      auxerre     strasbourg               h
9                   ligue 1  montpellier         rennes               h
10                  ligue 1       angers       toulouse               a
11                  serie b         bari      sampdoria               h
12               bundesliga     augsburg       freiburg         